[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_4_GIS_analytics/assignments/dsm_analysis.ipynb)

# Assignment 4A: Analysis of UAS data processing results

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

In this notebook we take DSMs of the same fields at Lake Wheeler processed in different ways and subtract them from each other: with and without ground control points (GCPs), in three software packages, and from two surveys four months apart. The differences show the errors introduced by the processing, the artifacts of each software, and the real change on the ground.

This notebook runs the GRASS half of the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_4_GIS_analytics/assignments/assignment_4a.html) in Python. The Metashape measurements (distances, contours, cross section, volume) are done in Metashape and are not part of this notebook. You can run it in Google Colab (badge above) or locally with GRASS 8.5 installed.

## Learning objectives

- Quantify and map the bowl effect: the systematic error in a DSM processed without GCPs
- Compare DSMs of one flight generated in Agisoft Metashape, Pix4D, and Trimble Business Center
- Use terrain analysis (slope, aspect, shaded relief) to expose distortions that are invisible in the elevation itself
- Map elevation change between two UAS surveys and separate real change from errors and artifacts

## Outline

- Part 1: Environment setup
- Part 2: Influence of GCPs
- Part 3: Software comparison
- Part 4: Terrain analysis of the DSMs
- Part 5: Change between two surveys
- Part 6: Your turn and the report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab).

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the three install cells.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 from conda-forge with mamba. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5

Check that GRASS is installed and which Python version is running.

In [ ]:
!grass --version

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5).

In [ ]:
import json
import os
import subprocess
from pathlib import Path

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### Download the data

The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters, about 225 MB) holds every DSM used in this notebook in its `PERMANENT` mapset:

- `sample_DSM` and `sample_DSM_noGCPs`: one flight processed with 4 GCPs and without
- `2015_06_20_DSM_agi_11GCP`, `2015_06_20_pix4d_11GCP_dsm`, `2015_06_20_DSM_Trimble_11GCP`: the flight of June 20, 2015 processed with 11 GCPs in Agisoft, Pix4D, and Trimble Business Center
- `2015_06_20_DSM_agi_noGCP`, `2015_06_20_DSM_pix4d_NoGCP`: the same flight without GCPs in Agisoft and Pix4D
- `2015_10_06_DSM_agi_8GCPs`: the flight of October 6, 2015, Agisoft, 8 GCPs
- `GCP_12_degrees`: the surveyed GCPs

An orthophoto for context is downloaded later, in Part 5, because it is large.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

PROJECT_URL = "https://storage.googleapis.com/gis-course-data/gis584/grassdata/Lake_Wheeler_NCspm.zip"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `analysis` inside the Lake Wheeler project for our work (the `PERMANENT` mapset stays untouched), start the session there, and create a `Tools` object. From here on, `tools.r_mapcalc(...)` runs `r.mapcalc`, `tools.g_region(...)` runs `g.region`, and so on; a flag like `-e` is passed as `flags="e"`.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "analysis"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session)

print(tools.g_proj(flags="p").text)
print(tools.g_list(type="raster,vector", mapset="PERMANENT", flags="m").text)

### Two helpers

Every comparison below produces a difference raster, gets a color table, and is summarized. Two small functions keep those cells short: `set_colors` writes `r.colors` rules from a string (the same rules you would type in the GUI's *Define* tab), and `describe` prints the statistics from `r.univar`.

In [ ]:
def set_colors(raster, rules):
    """Apply r.colors rules given as a multi-line string of 'value color' pairs."""
    gs.write_command("r.colors", map=raster, rules="-", stdin=rules, quiet=True)


def describe(raster, label=None):
    """Print the range, mean, median, and spread of a raster in the current region."""
    s = tools.r_univar(map=raster, flags="e", format="json").json
    print(f"{label or raster}: {s['n']:,} cells")
    print(f"  min {s['min']:8.2f}  max {s['max']:8.2f}  mean {s['mean']:8.2f}  median {s['median']:8.2f}  stddev {s['stddev']:6.2f} m")
    return s


# Rules for differences that should be near zero: -1 m to 1 m, white in the middle.
DIFF_RULES = """
-40 red
-1 orange
-0.5 yellow
-0.1 grey
0 white
0.1 grey
0.5 cyan
1 aqua
35 blue
"""

---
## Part 2: Influence of GCPs

### The bowl effect

Without ground control, the bundle adjustment has only the camera positions from the aircraft's GNSS receiver, accurate to a few meters, to fit the lens model against. The result is a DSM that has the right shape locally but is shifted in elevation and bent into a shallow bowl or dome over the whole block. Subtracting the DSM without GCPs from the DSM with GCPs makes the error visible.

`sample_DSM` was generated with 4 GCPs (UAV 3, 4, 6, and 8, the four inside its extent); `sample_DSM_noGCPs` is the same imagery without them. We set the computational region to the DSM and compute the difference with raster map algebra.

In [ ]:
tools.g_region(raster="sample_DSM")
tools.r_mapcalc(expression="GCP_noGCP = sample_DSM - sample_DSM_noGCPs", overwrite=True)

set_colors("GCP_noGCP", """
0 red
35 orange
36 yellow
37 cyan
38 aqua
39 blue
""")

gcp_stats = describe("GCP_noGCP", "With GCPs minus without GCPs")

The **mean** of the difference is the vertical shift: how far off the elevation would be, on average, if you trusted the DSM without GCPs. The **standard deviation** and the range between the center and the edges describe the bowl. Note both for every comparison; the report asks for them. The color rules are chosen to spread the colors over the 35 to 39 m band where most cells fall, so that the shape of the bowl, not the shift, dominates the map. Trees and buildings fall outside that band (the two surfaces reconstruct them differently) and show as red or white specks.

#### Map: Difference between the DSMs with and without GCPs, with the GCP locations

In [ ]:
gcp_map = gj.Map(width=600, height=550)
gcp_map.d_rast(map="GCP_noGCP")
gcp_map.d_vect(
    map="GCP_12_degrees@PERMANENT",
    attribute_column="GCP",
    icon="basic/circle",
    size=10,
    color="black",
    fill_color="yellow",
    label_color="black",
    label_size=10,
)
gcp_map.d_legend(raster="GCP_noGCP", at=(5, 40, 2, 6), title="Difference (m)", range=(35, 39))
gcp_map.d_barscale(at=(60, 5))
gcp_map.show()

### A profile through the bowl

`r.profile` samples the difference raster along a line. We draw it west to east through the center of the region and plot it, which shows the shape of the bowl in cross section.

In [ ]:
import matplotlib.pyplot as plt

region = tools.g_region(flags="g").keyval
center_n = (float(region["n"]) + float(region["s"])) / 2
west, east = float(region["w"]), float(region["e"])

profile = tools.r_profile(input="GCP_noGCP", coordinates=(west, center_n, east, center_n)).text
distance, value = [], []
for line in profile.splitlines():
    parts = line.split()
    if len(parts) == 2 and parts[1] != "*":
        distance.append(float(parts[0]))
        value.append(float(parts[1]))

plt.figure(figsize=(8, 3))
plt.plot(distance, value)
plt.xlabel("Distance along the west-east line (m)")
plt.ylabel("With GCPs minus without (m)")
plt.title("Profile of the difference through the center of the sample DSM")
plt.grid(True)
plt.show()

> **Think about it:** Where in the area are the GCPs, and where is the difference largest? With 4 GCPs the adjustment is pinned at 4 points; what happens to the surface between and beyond them?

---
## Part 3: Software comparison

### The bowl effect in two software packages

The flight of June 20, 2015 was processed with the same 11 GCPs in Agisoft Metashape, Pix4D, and Trimble Business Center, and without GCPs in Agisoft and Pix4D. First the bowl effect of each program: with minus without GCPs. The color rules differ because the two programs are off by very different amounts; the statistics tell you by how much.

In [ ]:
tools.g_region(raster="2015_06_20_DSM_agi_11GCP")

tools.r_mapcalc(expression="agi_GCP_agi_noGCP = 2015_06_20_DSM_agi_11GCP - 2015_06_20_DSM_agi_noGCP", overwrite=True)
set_colors("agi_GCP_agi_noGCP", """
0 red
35 orange
36 yellow
37 cyan
38 aqua
42 blue
""")

tools.r_mapcalc(expression="p4d_GCP_p4d_noGCP = 2015_06_20_pix4d_11GCP_dsm - 2015_06_20_DSM_pix4d_NoGCP", overwrite=True)
set_colors("p4d_GCP_p4d_noGCP", """
50 red
69 orange
70 yellow
71 cyan
72 aqua
75 blue
""")

agi_stats = describe("agi_GCP_agi_noGCP", "Agisoft: with GCPs minus without")
p4d_stats = describe("p4d_GCP_p4d_noGCP", "Pix4D: with GCPs minus without")

#### Map: Bowl effect in Agisoft (left) and Pix4D (right)

In [ ]:
for raster, title, legend_range in [
    ("agi_GCP_agi_noGCP", "Agisoft (m)", (30, 42)),
    ("p4d_GCP_p4d_noGCP", "Pix4D (m)", (66, 75)),
]:
    m = gj.Map(width=500, height=500)
    m.d_rast(map=raster)
    m.d_vect(map="GCP_12_degrees@PERMANENT", icon="basic/circle", size=8, color="black", fill_color="yellow")
    m.d_legend(raster=raster, at=(5, 40, 2, 6), title=title, range=legend_range)
    m.d_barscale(at=(60, 5))
    m.show()

> **Think about it:** The two color tables cover different ranges, so do not compare the maps by color alone. Use the statistics. Which software has the larger vertical shift (the mean)? Which one has the more curved bowl (the standard deviation)? These are two different questions and they do not have the same answer. Is the pattern a dome, a tilt, or both?

### The GCP-controlled products against each other

With GCPs, the three programs should agree. We subtract them pairwise and apply a color table from -1 m to 1 m, with white for differences within 10 cm. Cells outside the overlapping extent of a pair are null.

In [ ]:
pairs = {
    "agi_trimble": "2015_06_20_DSM_agi_11GCP - 2015_06_20_DSM_Trimble_11GCP",
    "p4d_agi": "2015_06_20_pix4d_11GCP_dsm - 2015_06_20_DSM_agi_11GCP",
    "p4d_trimble": "2015_06_20_pix4d_11GCP_dsm - 2015_06_20_DSM_Trimble_11GCP",
}
for name, expression in pairs.items():
    tools.r_mapcalc(expression=f"{name} = {expression}", overwrite=True)
    set_colors(name, DIFF_RULES)
    _ = describe(name)

#### Map: Agisoft minus Trimble, Pix4D minus Agisoft, Pix4D minus Trimble

In [ ]:
for name in pairs:
    m = gj.Map(width=500, height=500)
    m.d_rast(map=name)
    m.d_legend(raster=name, at=(5, 40, 2, 6), title=f"{name} (m)", range=(-1, 1))
    m.d_barscale(at=(60, 5))
    m.show()

The Trimble DSM covers a smaller area than the other two, so the maps that include it are null outside its extent.

> **Think about it:** The means are within a few centimeters of zero, but the maps are not blank. Where are the differences: along tree lines and buildings, at the edges of the block, in stripes along the flight lines? Which of these are artifacts of one program's interpolation, and which are differences in how the surface was reconstructed?

---
## Part 4: Terrain analysis of the DSMs

Terrain analysis shows the bowl more clearly than the elevation does: a dome becomes a slow gradient in slope and a radial pattern in aspect, on top of the real terraces and tree lines. We compute slope, aspect, and profile curvature with `r.slope.aspect` and a shaded relief with `r.relief` for the sample DSM with and without GCPs.

In [ ]:
tools.g_region(raster="sample_DSM")

for dsm, suffix in [("sample_DSM", "sample_DSM"), ("sample_DSM_noGCPs", "sample_DSM_noGCP")]:
    tools.r_slope_aspect(
        elevation=dsm,
        slope=f"{suffix}_slope",
        aspect=f"{suffix}_aspect",
        pcurvature=f"{suffix}_curv",
        overwrite=True,
    )
    tools.r_relief(input=dsm, output=f"{suffix}_relief", overwrite=True)

_ = describe("sample_DSM_slope", "Slope with GCPs (degrees)")
_ = describe("sample_DSM_noGCP_slope", "Slope without GCPs (degrees)")

#### Map: Aspect with GCPs (left) and without (right)

In [ ]:
for raster, title in [("sample_DSM_aspect", "Aspect, GCPs"), ("sample_DSM_noGCP_aspect", "Aspect, no GCPs")]:
    m = gj.Map(width=500, height=450)
    m.d_rast(map=raster)
    m.d_legend(raster=raster, at=(5, 40, 2, 6), title=title)
    m.d_barscale(at=(60, 5))
    m.show()

#### Map: Shaded relief with GCPs (left) and without (right)

In [ ]:
for raster, title in [("sample_DSM_relief", "Relief, GCPs"), ("sample_DSM_noGCP_relief", "Relief, no GCPs")]:
    m = gj.Map(width=500, height=450)
    m.d_rast(map=raster)
    m.d_barscale(at=(60, 5))
    m.show()

### Optional: addons for visualizing surfaces

Three GRASS addons are useful for seeing subtle features in a DSM: [r.local.relief](https://grass.osgeo.org/grass85/manuals/addons/r.local.relief.html) (local relief model), [r.shaded.pca](https://grass.osgeo.org/grass85/manuals/addons/r.shaded.pca.html) (principal components of shaded reliefs from several directions), and [r.skyview](https://grass.osgeo.org/grass85/manuals/addons/r.skyview.html) (sky view factor). `g.extension` downloads them from the GRASS addons repository; the cell below installs and runs `r.local.relief`. It needs network access and takes a minute.

In [ ]:
try:
    tools.g_extension(extension="r.local.relief")
    # The addon was installed after the Tools object was created, so call it with grass.script.
    for dsm, suffix in [("sample_DSM", "sample_DSM"), ("sample_DSM_noGCPs", "sample_DSM_noGCP")]:
        gs.run_command("r.local.relief", input=dsm, output=f"{suffix}_local_relief", overwrite=True)
    for raster in ("sample_DSM_local_relief", "sample_DSM_noGCP_local_relief"):
        m = gj.Map(width=500, height=450)
        m.d_rast(map=raster)
        m.d_legend(raster=raster, at=(5, 40, 2, 6), title=raster)
        m.show()
except Exception as error:  # the addon install can fail without network access
    print("r.local.relief is not available:", error)

---
## Part 5: Change between two surveys

### June to October 2015

The same fields were flown on June 20 and October 6, 2015, both processed in Agisoft with GCPs. Subtracting October from June gives elevation **loss** where the June surface was higher (harvested crops, removed material) and **gain** where October is higher (grown crops, deposited material). The roads and other bare, stable ground tell you the accuracy of the comparison: any difference there is error, not change.

In [ ]:
tools.g_region(raster="2015_06_20_DSM_agi_11GCP")
tools.r_mapcalc(expression="diff_jun_oct_agis = 2015_06_20_DSM_agi_11GCP - 2015_10_06_DSM_agi_8GCPs", overwrite=True)
set_colors("diff_jun_oct_agis", DIFF_RULES)
change_stats = describe("diff_jun_oct_agis", "June minus October 2015")

#### Map: June to October 2015 elevation change (positive: June higher)

In [ ]:
change_map = gj.Map(width=600, height=600)
change_map.d_rast(map="diff_jun_oct_agis")
change_map.d_vect(map="GCP_12_degrees@PERMANENT", icon="basic/circle", size=8, color="black", fill_color="yellow")
change_map.d_legend(raster="diff_jun_oct_agis", at=(5, 40, 2, 6), title="June - Oct (m)", range=(-1, 1))
change_map.d_barscale(at=(60, 5))
change_map.show()

### How much changed, and where

A classified map and an area table make the change countable. We call differences beyond half a meter loss or gain and everything in between stable, then let `r.report` sum the area of each class in hectares. Half a meter is well above the error seen on the roads; change the threshold if your reading of the roads suggests a different value.

In [ ]:
THRESHOLD = 0.5  # meters

tools.r_mapcalc(
    expression=f"change_class = if(diff_jun_oct_agis > {THRESHOLD}, 1, if(diff_jun_oct_agis < -{THRESHOLD}, 3, 2))",
    overwrite=True,
)
gs.write_command("r.category", map="change_class", rules="-", separator=":", stdin="1:loss (June higher)\n2:stable\n3:gain (October higher)")
set_colors("change_class", "1 orange\n2 white\n3 blue")

print(tools.r_report(map="change_class", units="h,p", flags="n").text)

### Orthophoto for context

To interpret the change you need to know what was on the fields. The course has orthophotos of Lake Wheeler from November 2015 (after the harvest), March 2017, and June 2017; none was flown on the dates of the two DSMs, but they show the land use. The cell downloads the November 2015 orthophoto as a GRASS pack (about 280 MB), restores it with `r.unpack`, and draws the classified change over it. Skip it if you are short on time or bandwidth.

In [ ]:
ORTHO_URL = "https://storage.googleapis.com/gis-course-data/gis584/uas-flight-data/Lake%20Wheeler%20-%20NCSU/112415/ortho_2015_11_24.pack"
!curl -sSL -o ortho_2015_11_24.pack {ORTHO_URL}
tools.r_unpack(input="ortho_2015_11_24.pack", flags="o", overwrite=True)

# Keep only the cells that changed so the orthophoto shows through elsewhere.
tools.r_mapcalc(expression="change_only = if(change_class == 2, null(), change_class)", overwrite=True)
tools.r_colors(map="change_only", raster="change_class")

ortho_map = gj.Map(width=600, height=600, use_region=True)  # zoom to the DSM region
ortho_map.d_rast(map="ortho_2015_11_24")
ortho_map.d_rast(map="change_only")
ortho_map.d_legend(raster="change_class", at=(5, 20, 2, 6), title="Change", bgcolor="white", flags="b")
ortho_map.d_barscale(at=(60, 5))
ortho_map.show()

> **Think about it:** Which of the loss and gain patches follow field boundaries (crops), which follow tree lines (canopy growth or leaf-off differences), and which follow flight lines or the edges of the block (processing artifacts)? What is the difference on the roads, and what does it say about the accuracy of a change map from two UAS surveys?

---
## Part 6: Your turn

The two exercises are optional and not required for the report.

### Exercise 1: Fewer GCPs

**Task:** The flight of March 18, 2015 was processed in Agisoft with 6 GCPs (`2015_03_18_DSM_agi_6GCP`) and without (`2015_03_18_DSM_agi_noGCP`). Compute the difference, give it a color table, map it with the GCPs, and compare its mean and range with the June flight (11 GCPs) from Part 3.

**Hint:** Set the region to the March DSM first. Run `describe` on the new raster and look at the 35 to 42 m rules used for the June flight; adjust them to the range you get.

In [ ]:
# Your code for Exercise 1 here
# tools.g_region(raster="2015_03_18_DSM_agi_6GCP")
# tools.r_mapcalc(expression="mar_GCP_noGCP = 2015_03_18_DSM_agi_6GCP - 2015_03_18_DSM_agi_noGCP", overwrite=True)
# describe("mar_GCP_noGCP")
# set_colors("mar_GCP_noGCP", ...)
# map it

### Exercise 2: Change over the growing season

**Task:** Repeat Part 5 for March to June 2015 (`2015_03_18_DSM_agi_6GCP` and `2015_06_20_DSM_agi_11GCP`): difference map, classified change, and the area of loss, stable, and gain in hectares. Does the sign of the change match what you expect between early spring and early summer?

**Hint:** Use new map names so you keep the June to October results. Both DSMs were processed with GCPs, so any difference on the roads is the combined error of two products.

In [ ]:
# Your code for Exercise 2 here
# tools.g_region(raster="2015_06_20_DSM_agi_11GCP")
# tools.r_mapcalc(expression="diff_mar_jun = 2015_06_20_DSM_agi_11GCP - 2015_03_18_DSM_agi_6GCP", overwrite=True)
# ...

---
## Report questions

Answer each question in the Markdown cell below it. These answers, with the maps from this notebook and your Metashape measurements, are the content of your report (see the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_4_GIS_analytics/assignments/assignment_4a.html) for the submission details).

#### Question 1: Measurements in Metashape

This part is done in Metashape, not in the notebook. Report the distance between UAV 9 and UAV 12 estimated by Metashape and calculated from the coordinates, with the calculation shown and the difference in meters and percent; the distance between the two cameras; the diameter and height of the sheaf; the cross section with its location; and the area and volume of the waste pile with the delimitation you used and the model parameters.

`Add text here with your answer to question 1`

#### Question 2: Influence of GCPs

Describe the difference between the DSMs with and without GCPs: the mean shift, the standard deviation, the shape of the bowl, and where it is largest. Include a slope, aspect, or relief map that shows the bowl. Explain the bowl effect and its shape, referring to the location of the four GCPs on your map.

`Add text here with your answer to question 2`

#### Question 3: Software comparison

Which software has the larger vertical shift, and which has the more curved bowl? Give the numbers. How do the three GCP-controlled products relate to each other (means, spread, where they disagree)? What artifacts can you recognize in the pairwise maps, and what causes them?

`Add text here with your answer to question 3`

#### Question 4: Change between surveys

From the June to October map and the area table, say which differences are real change in vegetation or bare ground, which are errors, and which are artifacts, and how you told them apart. Use the roads, the area table, and the orthophoto as evidence. State the error you read on the roads and the threshold you used.

`Add text here with your answer to question 4`